# ☀️ Solar Power Generation Prediction from Weather Data
**Task:** regression / time-series prediction of solar power output under changing environmental conditions.
**Dataset:** [Renewable Power Generation and Weather Conditions (Kaggle)](https://www.kaggle.com/datasets/pythonafroz/renewable-power-generation-and-weather-conditions)

**Workflow:** load data → EDA → feature engineering → chronological split → model comparison (KNN, Linear Regression, Random Forest, Gradient Boosting) → KNN tuning → feature importance → save best model.

## 1. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import config
from models_def import get_models, tune_knn
from utils import evaluate, load_data, prepare_data, time_split

sns.set_theme(style="whitegrid")

## 2. Load data

In [ ]:
df = load_data()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data quality

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 4. Feature engineering
Weather columns are kept as-is; calendar features (hour, month, day of year and sin/cos encodings) are added from the timestamp. Columns that hold *other* power outputs are dropped to avoid leakage.

In [ ]:
X, y, info = prepare_data(df)
print("Target      :", info["target"])
print("Time column :", info["time_col"])
print("Features    :", info["feature_names"])
print("Ignored     :", info["dropped_cols"])

## 5. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=40, kde=True, ax=axes[0]); axes[0].set_title(f"Distribution of {info['target']}")
if info["time_index"] is not None:
    axes[1].plot(info["time_index"].iloc[:24*14], y.iloc[:24*14])
    axes[1].set_title("First 14 days of generation")
plt.tight_layout(); plt.show()

In [ ]:
if info["time_index"] is not None:
    d = pd.DataFrame({"hour": X["hour"], "month": X["month"], "power": y})
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    d.groupby("hour")["power"].mean().plot(ax=axes[0], marker="o", title="Average power by hour of day")
    d.groupby("month")["power"].mean().plot.bar(ax=axes[1], title="Average power by month")
    plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(pd.concat([X[info["weather_cols"]], y], axis=1).corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation: weather variables vs. power"); plt.show()

## 6. Chronological train / test split
The last 20% of the timeline is held out. Shuffling would leak future weather patterns into training, so we do **not** shuffle.

In [ ]:
X_train, X_test, y_train, y_test = time_split(X, y)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 7. Model comparison

In [ ]:
fitted, rows, preds = {}, [], {}
for name, model in get_models().items():
    model.fit(X_train, y_train)
    fitted[name] = model
    preds[name] = model.predict(X_test)
    rows.append({"Model": name, **evaluate(y_test, preds[name])})
pd.DataFrame(rows).sort_values("RMSE").round(4)

## 8. Tune KNN
Best `k` and weighting are searched with `TimeSeriesSplit` cross-validation (always training on the past, validating on the future).

In [ ]:
grid = tune_knn(X_train, y_train)
print("Best params:", grid.best_params_)
fitted["KNN (tuned)"] = grid.best_estimator_
preds["KNN (tuned)"] = grid.best_estimator_.predict(X_test)
rows.append({"Model": "KNN (tuned)", **evaluate(y_test, preds["KNN (tuned)"])})
results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=results, x="RMSE", y="Model", color="#4C72B0")
plt.title("Model comparison (lower RMSE is better)"); plt.tight_layout(); plt.show()

## 9. Actual vs. predicted

In [ ]:
best_name = results.iloc[0]["Model"]
n = min(24 * 14, len(y_test))
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:n], label="Actual")
plt.plot(preds[best_name][:n], label=f"Predicted ({best_name})")
plt.legend(); plt.title("Actual vs predicted solar power"); plt.show()

plt.figure(figsize=(5, 5))
plt.scatter(y_test, preds[best_name], s=4, alpha=0.4)
lim = [0, max(y_test.max(), preds[best_name].max())]
plt.plot(lim, lim, "r--"); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.show()

## 10. Feature importance (Random Forest)

In [ ]:
rf = fitted["Random Forest"].named_steps["model"]
pd.Series(rf.feature_importances_, index=info["feature_names"]).sort_values().plot.barh(figsize=(8, 5), color="#55A868")
plt.title("Which features drive solar power?"); plt.show()

## 11. Best model summary
To save the deployable model bundle used by the Streamlit app (`app.py`), run `python train.py`.

In [ ]:
import os; os.makedirs("models", exist_ok=True)
print("Best model:", best_name)
print(results.iloc[0].to_dict())

## 12. Conclusions
- Compare the table above: tree ensembles usually beat KNN and Linear Regression because the relationship between weather and solar output is non-linear (zero at night, saturating at midday).
- Time-of-day and cloud/radiation-type features typically dominate importance — see section 10.
- Next steps: add lag features (previous hours' power), try XGBoost/LightGBM, or an LSTM for sequence modelling.